# ATEM-CVD Model Training Guide
## Adaptive Transformer Ensemble Multimodal for Cardiovascular Disease
### Target Accuracy: 90%+

---

## 📋 STEP 1: Install Required Libraries

Run this first to install all dependencies:

In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install pandas numpy scikit-learn matplotlib seaborn
!pip install wfdb scipy
!pip install xgboost lightgbm catboost
!pip install shap lime
!pip install imbalanced-learn
!pip install pywavelets
!pip install optuna
!pip install wandb  # Optional: for experiment tracking

print("✅ All libraries installed successfully!")

'pip' is not recognized as an internal or external command,
operable program or batch file.
'pip' is not recognized as an internal or external command,
operable program or batch file.
'pip' is not recognized as an internal or external command,
operable program or batch file.
'pip' is not recognized as an internal or external command,
operable program or batch file.
'pip' is not recognized as an internal or external command,
operable program or batch file.
'pip' is not recognized as an internal or external command,
operable program or batch file.
'pip' is not recognized as an internal or external command,
operable program or batch file.
'pip' is not recognized as an internal or external command,
operable program or batch file.


✅ All libraries installed successfully!


'pip' is not recognized as an internal or external command,
operable program or batch file.


## 📂 STEP 2: Mount Drive and Import Libraries

**For Google Colab:**

In [2]:
# Uncomment for Google Colab
# from google.colab import drive
# drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import wfdb
import ast
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import pywt
import shap
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Using device: {device}")

ModuleNotFoundError: No module named 'imblearn'

## 📊 STEP 3: Load and Prepare PTB-XL Dataset

In [ ]:

# Set your data path dynamically (supports Kaggle and kagglehub download)
import os, ast


def find_first_file(search_roots, filename):
    for base in search_roots:
        if base and os.path.exists(base):
            for root, _, files in os.walk(base):
                if filename in files:
                    return os.path.join(root, filename)
    return None


def try_download_ptbxl():
    try:
        import kagglehub
        print("Attempting kagglehub download: likith012/ecgnet-ptb-xl ...")
        dl_path = kagglehub.dataset_download("likith012/ecgnet-ptb-xl")
        print(f"Download complete: {dl_path}")
        return dl_path
    except Exception as e:
        print(f"⚠️  kagglehub download failed: {e}")
        return None


search_roots = []
if os.path.exists('/kaggle/input/'):
    print("🔍 Running on Kaggle")
    print()
    print("📂 Available Kaggle input datasets:")
    for d in os.listdir('/kaggle/input/'):
        print(f"   /kaggle/input/{d}/")
    print()
    search_roots.append('/kaggle/input/')
else:
    print("🔍 Running locally")

ptbxl_csv = find_first_file(search_roots, 'ptbxl_database.csv')

if not ptbxl_csv:
    dl_root = try_download_ptbxl()
    if dl_root:
        search_roots.append(dl_root)
        ptbxl_csv = find_first_file([dl_root], 'ptbxl_database.csv')

if not ptbxl_csv:
    raise FileNotFoundError("ptbxl_database.csv not found in Kaggle inputs or kagglehub download. Please attach the PTB-XL dataset (likith012/ecgnet-ptb-xl).")

DATA_PATH = os.path.dirname(ptbxl_csv) + '/'
print(f"✅ PTB-XL found at: {DATA_PATH}")

# Load database
df = pd.read_csv(DATA_PATH + 'ptbxl_database.csv', index_col='ecg_id')
df.scp_codes = df.scp_codes.apply(lambda x: ast.literal_eval(x))

# Load SCP statements
agg_df = pd.read_csv(DATA_PATH + 'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]


def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))


# Apply diagnostic aggregation
df['diagnostic_superclass'] = df.scp_codes.apply(aggregate_diagnostic)

# Filter for binary classification (NORM vs others)
df['label'] = df['diagnostic_superclass'].apply(
    lambda x: 0 if 'NORM' in x else 1 if len(x) > 0 else -1
)
df = df[df['label'] != -1]

print(f"✅ Dataset loaded: {len(df)} samples")
print(f"   Normal: {sum(df['label'] == 0)} samples")
print(f"   Disease: {sum(df['label'] == 1)} samples")


## 🔧 STEP 4: Advanced ECG Signal Preprocessing with Wavelet Transform

In [ ]:
def load_ecg_signal(filename, sampling_rate=100):
    """Load ECG signal from PTB-XL dataset"""
    if sampling_rate == 100:
        data = wfdb.rdsamp(DATA_PATH + filename)
    else:
        data = wfdb.rdsamp(DATA_PATH + filename.replace('_hr', '_lr'))
    return data[0]

def wavelet_denoise(signal, wavelet='db4', level=1):
    """Denoise ECG signal using wavelet transform"""
    coeff = pywt.wavedec(signal, wavelet, mode="per")
    sigma = (1/0.6745) * np.median(np.abs(coeff[-level] - np.median(coeff[-level])))
    uthresh = sigma * np.sqrt(2 * np.log(len(signal)))
    coeff[1:] = (pywt.threshold(i, value=uthresh, mode='soft') for i in coeff[1:])
    return pywt.waverec(coeff, wavelet, mode='per')

def preprocess_ecg(signal):
    """Advanced preprocessing pipeline"""
    # Baseline wander removal
    signal = signal - np.mean(signal, axis=0)
    
    # Wavelet denoising for each lead
    denoised = np.zeros_like(signal)
    for i in range(signal.shape[1]):
        denoised[:, i] = wavelet_denoise(signal[:, i])
    
    # Normalization
    denoised = (denoised - np.mean(denoised, axis=0)) / (np.std(denoised, axis=0) + 1e-8)
    
    return denoised

print("✅ Preprocessing functions ready!")

## 🧬 STEP 5: Create Custom PyTorch Dataset

In [ ]:
class PTBXLDataset(Dataset):
    def __init__(self, dataframe, sampling_rate=100, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.sampling_rate = sampling_rate
        self.augment = augment
        
    def __len__(self):
        return len(self.df)
    
    def augment_signal(self, signal):
        """Data augmentation for ECG"""
        if np.random.rand() > 0.5:
            # Time scaling
            scale = np.random.uniform(0.9, 1.1)
            signal = signal * scale
        
        if np.random.rand() > 0.5:
            # Add Gaussian noise
            noise = np.random.normal(0, 0.01, signal.shape)
            signal = signal + noise
        
        return signal
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load and preprocess ECG
        filename = row['filename_hr'] if self.sampling_rate == 500 else row['filename_lr']
        signal = load_ecg_signal(filename, self.sampling_rate)
        signal = preprocess_ecg(signal)
        
        if self.augment:
            signal = self.augment_signal(signal)
        
        # Extract clinical features
        clinical_features = [
            row['age'],
            row['sex'],
            row['height'],
            row['weight'],
        ]
        
        # Convert to tensors
        ecg_tensor = torch.FloatTensor(signal.T)  # Shape: (12, 1000)
        clinical_tensor = torch.FloatTensor(clinical_features)
        label = torch.LongTensor([row['label']])[0]
        
        return ecg_tensor, clinical_tensor, label

print("✅ Dataset class created!")

## 🏗️ STEP 6: Build ResNet1D with Attention for ECG Feature Extraction

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.query = nn.Linear(in_dim, in_dim // 8)
        self.key = nn.Linear(in_dim, in_dim // 8)
        self.value = nn.Linear(in_dim, in_dim)
        self.gamma = nn.Parameter(torch.zeros(1))
        
    def forward(self, x):
        batch, channels, length = x.size()
        x_flat = x.permute(0, 2, 1)  # (B, L, C)
        
        Q = self.query(x_flat)  # (B, L, C//8)
        K = self.key(x_flat)    # (B, L, C//8)
        V = self.value(x_flat)  # (B, L, C)
        
        attention = F.softmax(torch.bmm(Q, K.permute(0, 2, 1)), dim=-1)
        out = torch.bmm(attention, V)
        out = out.permute(0, 2, 1)  # (B, C, L)
        
        return self.gamma * out + x

class ResBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=7, stride=stride, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=7, padding=3, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )
    
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class ResNet1DAttention(nn.Module):
    def __init__(self, input_channels=12, num_classes=2):
        super().__init__()
        
        # Initial convolution
        self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=15, stride=2, padding=7, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        
        # ResNet blocks
        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        
        # Attention
        self.attention = SelfAttention(256)
        
        # Global average pooling
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        
        # Feature output
        self.feature_dim = 256
        
    def _make_layer(self, in_channels, out_channels, blocks, stride):
        layers = []
        layers.append(ResBlock1D(in_channels, out_channels, stride))
        for _ in range(1, blocks):
            layers.append(ResBlock1D(out_channels, out_channels))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        
        x = self.attention(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        
        return x

print("✅ ResNet1D with Attention created!")

## 🧠 STEP 7: Build Transformer-Based Fusion Model

In [ ]:
class MultimodalTransformer(nn.Module):
    def __init__(self, ecg_dim=256, clinical_dim=4, num_classes=2, num_heads=8, num_layers=4):
        super().__init__()
        
        # ECG feature extractor
        self.ecg_encoder = ResNet1DAttention()
        
        # Clinical feature processor
        self.clinical_fc = nn.Sequential(
            nn.Linear(clinical_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        # Fusion layer
        fusion_dim = ecg_dim + 128
        self.fusion = nn.Linear(fusion_dim, 512)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=512,
            nhead=num_heads,
            dim_feedforward=2048,
            dropout=0.1,
            activation='gelu',
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, ecg, clinical):
        # Extract ECG features
        ecg_features = self.ecg_encoder(ecg)
        
        # Process clinical features
        clinical_features = self.clinical_fc(clinical)
        
        # Multimodal fusion
        fused = torch.cat([ecg_features, clinical_features], dim=1)
        fused = F.relu(self.fusion(fused))
        
        # Add sequence dimension for transformer
        fused = fused.unsqueeze(1)  # (B, 1, 512)
        
        # Transformer encoding
        transformed = self.transformer(fused)
        transformed = transformed.squeeze(1)  # (B, 512)
        
        # Classification
        output = self.classifier(transformed)
        
        return output, ecg_features

print("✅ Multimodal Transformer created!")

## 🎯 STEP 8: Training Functions with Focal Loss

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for ecg, clinical, labels in train_loader:
        ecg, clinical, labels = ecg.to(device), clinical.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs, _ = model(ecg, clinical)
        loss = criterion(outputs, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(train_loader), 100. * correct / total

def validate_epoch(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for ecg, clinical, labels in val_loader:
            ecg, clinical, labels = ecg.to(device), clinical.to(device), labels.to(device)
            
            outputs, _ = model(ecg, clinical)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            probs = F.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
    
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    auc = roc_auc_score(all_labels, all_probs)
    
    return total_loss / len(val_loader), accuracy * 100, f1 * 100, auc * 100

print("✅ Training functions ready!")

## 🚀 STEP 9: Main Training Loop with Patient-Wise Cross-Validation

In [ ]:
# Configuration
BATCH_SIZE = 64
EPOCHS = 50
LEARNING_RATE = 0.0001
N_FOLDS = 5

# Prepare data splits
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['label'])):
    print(f"\n{'='*60}")
    print(f"🔄 Training Fold {fold + 1}/{N_FOLDS}")
    print(f"{'='*60}\n")
    
    # Create datasets
    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]
    
    train_dataset = PTBXLDataset(train_df, sampling_rate=100, augment=True)
    val_dataset = PTBXLDataset(val_df, sampling_rate=100, augment=False)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # Initialize model
    model = MultimodalTransformer().to(device)
    criterion = FocalLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    # Training loop
    best_acc = 0
    patience = 0
    max_patience = 10
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [], 'val_f1': [], 'val_auc': []
    }
    
    for epoch in range(EPOCHS):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, val_f1, val_auc = validate_epoch(model, val_loader, criterion, device)
        scheduler.step()
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        history['val_auc'].append(val_auc)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | "
              f"Val F1: {val_f1:.2f}% | Val AUC: {val_auc:.2f}%")
        
        # Save best model
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f'best_model_fold{fold+1}.pth')
            patience = 0
        else:
            patience += 1
        
        # Early stopping
        if patience >= max_patience:
            print(f"\n⚠️ Early stopping at epoch {epoch+1}")
            break
    
    fold_results.append({
        'fold': fold + 1,
        'best_acc': best_acc,
        'final_f1': val_f1,
        'final_auc': val_auc,
        'history': history
    })
    
    print(f"\n✅ Fold {fold+1} Best Accuracy: {best_acc:.2f}%\n")

# Print overall results
print("\n" + "="*60)
print("📊 CROSS-VALIDATION RESULTS")
print("="*60)
avg_acc = np.mean([r['best_acc'] for r in fold_results])
avg_f1 = np.mean([r['final_f1'] for r in fold_results])
avg_auc = np.mean([r['final_auc'] for r in fold_results])

print(f"\n🎯 Average Accuracy: {avg_acc:.2f}% (±{np.std([r['best_acc'] for r in fold_results]):.2f})")
print(f"🎯 Average F1-Score: {avg_f1:.2f}% (±{np.std([r['final_f1'] for r in fold_results]):.2f})")
print(f"🎯 Average AUC-ROC: {avg_auc:.2f}% (±{np.std([r['final_auc'] for r in fold_results]):.2f})")

if avg_acc >= 90:
    print("\n🎉 TARGET ACHIEVED: 90%+ ACCURACY! 🎉")
else:
    print(f"\n⚠️ Current: {avg_acc:.2f}% | Target: 90%+ | Gap: {90-avg_acc:.2f}%")

## 📈 STEP 10: Visualize Training Results

In [ ]:
# Plot training history for best fold
best_fold = max(fold_results, key=lambda x: x['best_acc'])
history = best_fold['history']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss
axes[0, 0].plot(history['train_loss'], label='Train Loss')
axes[0, 0].plot(history['val_loss'], label='Val Loss')
axes[0, 0].set_title('Loss Curve')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Accuracy
axes[0, 1].plot(history['train_acc'], label='Train Accuracy')
axes[0, 1].plot(history['val_acc'], label='Val Accuracy')
axes[0, 1].axhline(y=90, color='r', linestyle='--', label='90% Target')
axes[0, 1].set_title('Accuracy Curve')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy (%)')
axes[0, 1].legend()
axes[0, 1].grid(True)

# F1 Score
axes[1, 0].plot(history['val_f1'], label='Val F1', color='green')
axes[1, 0].set_title('F1 Score')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('F1 Score (%)')
axes[1, 0].legend()
axes[1, 0].grid(True)

# AUC-ROC
axes[1, 1].plot(history['val_auc'], label='Val AUC', color='purple')
axes[1, 1].set_title('AUC-ROC Score')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('AUC (%)')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('training_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Training curves saved as 'training_results.png'")

## 🔍 STEP 11: Extract Features for XGBoost Ensemble

In [ ]:
def extract_features(model, dataloader, device):
    """Extract deep features for ensemble learning"""
    model.eval()
    all_features = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for ecg, clinical, labels in dataloader:
            ecg, clinical = ecg.to(device), clinical.to(device)
            outputs, features = model(ecg, clinical)
            probs = F.softmax(outputs, dim=1)
            
            # Combine deep features with clinical data
            combined_features = torch.cat([features, clinical], dim=1)
            
            all_features.append(combined_features.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
    
    return np.vstack(all_features), np.concatenate(all_labels), np.vstack(all_probs)

print("✅ Feature extraction function ready!")

## 🌳 STEP 12: Train XGBoost Meta-Learner (Ensemble)

In [ ]:
# Load best model from first fold
best_model = MultimodalTransformer().to(device)
best_model.load_state_dict(torch.load('best_model_fold1.pth'))

# Extract features from validation set
val_dataset = PTBXLDataset(df.iloc[val_idx], sampling_rate=100, augment=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

X_features, y_true, dl_probs = extract_features(best_model, val_loader, device)

# Add DL predictions as features
X_ensemble = np.hstack([X_features, dl_probs])

# Train XGBoost meta-learner
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)

# Split for meta-learning
from sklearn.model_selection import train_test_split
X_train_meta, X_val_meta, y_train_meta, y_val_meta = train_test_split(
    X_ensemble, y_true, test_size=0.2, random_state=42, stratify=y_true
)

print("\n🌳 Training XGBoost Meta-Learner...")
xgb_model.fit(
    X_train_meta, y_train_meta,
    eval_set=[(X_val_meta, y_val_meta)],
    early_stopping_rounds=20,
    verbose=10
)

# Evaluate ensemble
y_pred_meta = xgb_model.predict(X_val_meta)
y_prob_meta = xgb_model.predict_proba(X_val_meta)[:, 1]

ensemble_acc = accuracy_score(y_val_meta, y_pred_meta) * 100
ensemble_f1 = f1_score(y_val_meta, y_pred_meta, average='weighted') * 100
ensemble_auc = roc_auc_score(y_val_meta, y_prob_meta) * 100

print("\n" + "="*60)
print("🎯 ENSEMBLE MODEL RESULTS")
print("="*60)
print(f"Accuracy: {ensemble_acc:.2f}%")
print(f"F1-Score: {ensemble_f1:.2f}%")
print(f"AUC-ROC: {ensemble_auc:.2f}%")

if ensemble_acc >= 90:
    print("\n🎉🎉🎉 90%+ ACCURACY ACHIEVED WITH ENSEMBLE! 🎉🎉🎉")

# Save XGBoost model
xgb_model.save_model('xgboost_meta_learner.json')
print("\n✅ Ensemble model saved!")

## 🔬 STEP 13: SHAP Explainability Analysis

In [ ]:
# SHAP analysis for XGBoost
print("\n🔬 Computing SHAP values...")
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_val_meta[:100])  # Sample for speed

# Summary plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_val_meta[:100], plot_type="bar", show=False)
plt.title('Feature Importance (SHAP Values)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ SHAP analysis complete! Saved as 'shap_importance.png'")

## 📊 STEP 14: Generate Confusion Matrix and Classification Report

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_val_meta, y_pred_meta)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Normal', 'Disease'],
            yticklabels=['Normal', 'Disease'])
plt.title('Confusion Matrix - Ensemble Model', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Classification Report
print("\n" + "="*60)
print("📋 CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_val_meta, y_pred_meta, 
                          target_names=['Normal', 'Disease']))

print("\n✅ All visualizations saved!")

## 💾 STEP 15: Save Final Model and Results

In [ ]:
# Save all results
results_summary = {
    'cross_validation': {
        'avg_accuracy': avg_acc,
        'avg_f1': avg_f1,
        'avg_auc': avg_auc,
        'fold_results': fold_results
    },
    'ensemble': {
        'accuracy': ensemble_acc,
        'f1_score': ensemble_f1,
        'auc_roc': ensemble_auc
    }
}

import json
with open('training_results.json', 'w') as f:
    json.dump(results_summary, f, indent=4)

print("\n✅ All results saved!")
print("\n📁 Files created:")
print("   • best_model_fold1.pth - fold5.pth (Deep Learning models)")
print("   • xgboost_meta_learner.json (XGBoost ensemble)")
print("   • training_results.png (Training curves)")
print("   • shap_importance.png (Feature importance)")
print("   • confusion_matrix.png (Confusion matrix)")
print("   • training_results.json (Performance metrics)")
print("\n🎉 Training complete!")